# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading and analyzing the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. The dataset provides survey responses on mental health indicators, demographic factors, and assessment scores (GAD-7, PHQ-9, PSQ) from residents of Kilifi County, Kenya.

### Dataset Source
The dataset source is provided via the following Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed (run this cell if you are missing the library)
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {metadata.version if hasattr(metadata, 'version') else 'N/A'}")
print(f"Date Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"License: {metadata.license if hasattr(metadata, 'license') else 'N/A'}")

## 2. Data Overview
Review available record sets and fields.

**Note**: We will reference all entities by their Croissant `@id` fields, as per best practice.

Let's inspect the available record sets in the metadata. We'll display their IDs, names, and the associated fields and columns.

In [ ]:
# List all the available record sets and their fields
record_sets = metadata.recordSets
print("Available record sets and their fields (by @id):\n")
record_set_ids = []

for rs in record_sets:
    print(f"RecordSet @id: {rs.id if hasattr(rs, 'id') else rs.__dict__.get('@id', 'N/A')}")
    record_set_ids.append(rs.id if hasattr(rs, 'id') else rs.__dict__.get('@id', 'N/A'))
    print(f"  Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields (by @id):")
        for field in rs.fields:
            print(f"    - {field.id if hasattr(field, 'id') else field.__dict__.get('@id', 'N/A')} (name: {getattr(field, 'name', 'N/A')}, type: {getattr(field, 'dataType', 'N/A')})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns (by @id):")
        for col in rs.columns:
            print(f"    - {col.id if hasattr(col, 'id') else col.__dict__.get('@id', 'N/A')} (name: {getattr(col, 'name', 'N/A')}, type: {getattr(col, 'dataType', 'N/A')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll extract data from all discoverable record sets. Be sure to reference the record set and field `@id`s from the overview above.

> **If no record sets exist, the list will be empty and the next cell may need to be customized with accurate record set IDs once known.**

In [ ]:
# Define the record set IDs to extract (as discovered above)
# If the record set list is empty, replace with known @ids, e.g., record_set_ids = ['cr:KilifiSurveyRecords']
# For demonstration, we proceed assuming at least one record set is present
if len(record_set_ids) == 0:
    print("No record sets discovered in the metadata. Please check the dataset schema or update with correct record set IDs.")
else:
    dataframes = {}
    for rs_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded record set {rs_id}. Shape: {df.shape}")
        except Exception as e:
            print(f"Could not load record set {rs_id}: {e}")

    # Preview the first DataFrame if any loaded
    if dataframes:
        first_rs_id = list(dataframes.keys())[0]
        print(f"\nAvailable columns in record set {first_rs_id}:")
        print(dataframes[first_rs_id].columns.tolist())
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's process the main survey data:
- Select a numeric field like the GAD-7 score or PHQ-9 score based on field `@id`.
- Filter records where this score is above a threshold.
- Normalize the numeric field.
- Group records by a demographic field (e.g., gender or age group).

**Identify field `@id`s** from the 'Data Overview' output above and adjust as needed below.

In [ ]:
if dataframes:
    # Pick the first record set for demonstration
    record_set_id = list(dataframes.keys())[0]

    # Try to pick common field names. Adjust if your schema differs.
    df = dataframes[record_set_id]
    print(f"Analyzing record set '{record_set_id}' with columns: {df.columns.tolist()}")

    # Try matching common numeric field by likely GAD or PHQ field @id or column name
    possible_gad_fields = [col for col in df.columns if 'gad' in col.lower() or 'GAD' in col]
    possible_phq_fields = [col for col in df.columns if 'phq' in col.lower() or 'PHQ' in col]
    numeric_field = possible_gad_fields[0] if possible_gad_fields else (possible_phq_fields[0] if possible_phq_fields else None)

    if numeric_field and numeric_field in df.columns:
        print(f"Using numeric field '@id': {numeric_field}")
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold} (count: {len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' (first 5):")
        display(filtered_df[[numeric_field, col_norm]].head())

        # Group by a demographic field (attempt 'gender' or 'age_group' fields, case-insensitive)
        group_fields = [col for col in df.columns if col.lower() in ('gender', 'sex', 'age_group', 'agegroup')]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nMean {numeric_field} by {group_field} (filtered records):")
            display(grouped_df)
        else:
            print("No demographic group field (e.g., gender or age group) found for grouping.")
    else:
        print("No appropriate numeric field (GAD/PHQ) found. Adjust the field @id as needed.")
else:
    print("No dataframes found. Extraction may have failed or no record sets defined.")

## 5. Visualization

Let's plot the distribution of the main numeric field (e.g., GAD-7 or PHQ-9 score), and the group means if grouped.

In [ ]:
if dataframes and numeric_field:
    df = dataframes[record_set_id]
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=20, color='#007bff', alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df exists from EDA, make a bar plot
    if 'grouped_df' in locals():
        plt.figure(figsize=(7,4))
        plt.bar(grouped_df[group_field], grouped_df[numeric_field], color='#28a745', alpha=0.7)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No suitable data or numeric field for visualization.")

## 6. Conclusion

- We demonstrated how to load a Croissant dataset using `mlcroissant` via a schema URL.
- The notebook provided an overview of available record sets and their fields by Croissant `@id`.
- We extracted, filtered, and normalized survey data (e.g. GAD-7/PHQ-9 scores), grouped by demographic features, and visualized distributions.
- For further analysis, refer to field and record set `@id`s in the metadata to ensure reproducibility across Croissant-conformant datasets.